El objetivo de este cuaderno es hacer un ensemble entre el mejor modelo de los cuadernos *LLM.ipynb* y *Transformers.ipynb*, junto con un modelo que trabaje con imágenes y otro modelo que trabaje con videos.

En este cuaderno también haremos un análisis de errores.

Dentro de los diferentes tipos de ensembles que podemos hacer nos decantamos por la opción de *Predicción estática* ya que es la opción que se suele hacer en las competiciones dentro del mundo de NLP y en la ciencia de datos porque nos permite ajustes los pesos miles de veces en un segundo sin tener que volver a tirar de tarjeta gráfica para procesar los videos de nuevo.

Los modelos que componen el ensemble son:
- *Mistral* (70% de peso) que ha sido el mejor modelo de entre los Transformers y LLM.
- *Google-in21k* (15% de peso) como modelo de imágenes.
- *VideoMAE* (15% de peso) como modelo de video.

Los pesos los justificamos por el rendimiento que nos han dado los modelos a la hora de realizar los entrenamientos.

In [1]:
!pip install decord
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 146.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import os
from tqdm import tqdm
from datasets import Dataset, Image

In [3]:
from peft import PeftModel
from PIL import Image as PILImage
from decord import VideoReader, cpu
import decord

In [4]:
# Dependencias específicas de modelos
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    ViTImageProcessor,
    ViTForImageClassification,
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification
)

In [5]:
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Definición de las rutas de los archivos

Definiendo cuáles son las rutas de los archivos .csv sobre los que vamos a hacer el ensemble

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Rutas de datos
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/dataset_imagenes_test.csv"
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/"

In [8]:
# Rutas donde tengo guardados los modelos
DIR_MODELO_TEXTO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565"
DIR_MODELO_IMAGEN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/ViT_FineTuned/modelo_final"
DIR_MODELO_VIDEO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/VideoMAE_FineTuned/modelo_final"

In [9]:
# Rutas para guardar las predicciones temporales y resultados
DIR_RESULTADOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble"
os.makedirs(DIR_RESULTADOS, exist_ok=True)
RUTA_PRED_TEXTO = os.path.join(DIR_RESULTADOS, "predicciones_test_mistral.csv")
RUTA_PRED_IMAGEN = os.path.join(DIR_RESULTADOS, "predicciones_test_vit.csv")
RUTA_PRED_VIDEO = os.path.join(DIR_RESULTADOS, "predicciones_test_videomae.csv")

# 2. Carga del dataset de test (común para todos)

In [10]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

#test_df

Cargando el dataset de test fijo...


# 3. Fase de generación de predicciones

+# 3.1. Predicciones de texto (*Mistral*)

In [11]:
if not os.path.exists(RUTA_PRED_TEXTO):
    print("\n--- Generando predicciones de Texto (Mistral) ---")

    # 1. Cargar Tokenizador y Modelo Base
    model_id = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=2, device_map="auto", torch_dtype=torch.bfloat16
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # 2. Cargar Pesos QLoRA
    model = PeftModel.from_pretrained(base_model, DIR_MODELO_TEXTO)
    model.eval()

    predicciones_texto = []

    # 3. Inferencia
    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia Mistral"):
        inputs = tokenizer(row["text"], return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # Aplicamos Softmax para obtener la probabilidad de clase 1
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

        predicciones_texto.append({
            "id_EXIST": row["id_EXIST"],
            "prob_texto": prob_misogino
        })

    pd.DataFrame(predicciones_texto).to_csv(RUTA_PRED_TEXTO, index=False)

    # Limpiamos memoria
    del model, base_model, tokenizer, inputs, outputs
    torch.cuda.empty_cache()
    print("Predicciones de texto guardadas.")


--- Generando predicciones de Texto (Gemma) ---


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Inferencia Mistral: 100%|██████████| 502/502 [00:49<00:00, 10.18it/s]


Predicciones de texto guardadas.


## 3.2. Predicciones imagen (*ViT + Max-Pooling*)

In [12]:
if not os.path.exists(RUTA_PRED_IMAGEN):
    print("\n--- Generando predicciones de Imagen (ViT) ---")

    processor_img = ViTImageProcessor.from_pretrained(DIR_MODELO_IMAGEN)
    model_img = ViTForImageClassification.from_pretrained(DIR_MODELO_IMAGEN).to(device)
    model_img.eval()

    df_imagenes = pd.read_csv(CSV_IMAGENES)
    test_ids = test_df['id_EXIST'].unique()
    test_img_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

    predicciones_por_video = {}

    for index, row in tqdm(test_img_df.iterrows(), total=len(test_img_df), desc="Inferencia ViT (Frames)"):
        id_vid = row['id_EXIST']
        try:
            image = PILImage.open(row['path_imagen']).convert("RGB")
            inputs = processor_img(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model_img(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

            if id_vid not in predicciones_por_video:
                predicciones_por_video[id_vid] = []
            predicciones_por_video[id_vid].append(prob_misogino)
        except Exception as e:
            continue

    # Aplicar Max-Pooling
    predicciones_img_final = []
    for id_vid, probs in predicciones_por_video.items():
        prob_max = max(probs) if probs else 0.0
        predicciones_img_final.append({
            "id_EXIST": id_vid,
            "prob_imagen": prob_max
        })

    pd.DataFrame(predicciones_img_final).to_csv(RUTA_PRED_IMAGEN, index=False)

    del model_img, processor_img, inputs, outputs
    torch.cuda.empty_cache()
    print("Predicciones de imagen guardadas.")

## 3.3. Predicciones de vídeo (*VideoMAE*)

In [13]:
if not os.path.exists(RUTA_PRED_VIDEO):
    print("\n--- Generando predicciones de Vídeo (VideoMAE) ---")

    processor_vid = VideoMAEImageProcessor.from_pretrained(DIR_MODELO_VIDEO)
    model_vid = VideoMAEForVideoClassification.from_pretrained(DIR_MODELO_VIDEO).to(device)
    model_vid.eval()

    def sample_frame_indices(clip_len, total_frames):
        if total_frames <= clip_len:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
        else:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

    predicciones_video = []

    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia VideoMAE"):
        id_vid = row["id_EXIST"]
        # Construir ruta del mp4 (Ajusta la lógica si path_video difiere)
        val = str(row.get('path_video', id_vid))
        if not val.endswith(".mp4"): val += ".mp4"
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, val)

        prob_misogino = 0.0
        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(16, len(vr))
            frames = vr.get_batch(frame_indices).numpy()
            inputs = processor_vid(list(frames), return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model_vid(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()
        except Exception as e:
            # Si el video falla, dejamos probabilidad neutra o 0
            prob_misogino = 0.5

        predicciones_video.append({
            "id_EXIST": id_vid,
            "prob_video": prob_misogino
        })

    pd.DataFrame(predicciones_video).to_csv(RUTA_PRED_VIDEO, index=False)

    del model_vid, processor_vid
    try: del inputs, outputs
    except: pass
    torch.cuda.empty_cache()
    print("Predicciones de vídeo guardadas.")

# 4. Ensemble multimodal

In [14]:
print("\n--- Ejecutando Ensemble ---")
df_mistral = pd.read_csv(RUTA_PRED_TEXTO)
df_vit = pd.read_csv(RUTA_PRED_IMAGEN)
df_videomae = pd.read_csv(RUTA_PRED_VIDEO)

# Unimos todo usando el 'id_EXIST'
df_ensemble = test_df[['id_EXIST', 'label', 'text']].merge(df_mistral, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_vit, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_videomae, on="id_EXIST", how="left")


--- Ejecutando Ensemble ---


In [15]:
# Rellenar posibles NaNs con 0.5 (incertidumbre) en caso de que algún modelo fallara en un id
df_ensemble = df_ensemble.fillna(0.5)

In [16]:
# DEFINICIÓN DE PESOS (Basado en el rendimiento individual de validación)
PESO_TEXTO = 0.70    # Mistral es muy superior
PESO_IMAGEN = 0.15   # ViT aporta contexto estático
PESO_VIDEO = 0.15    # VideoMAE aporta contexto temporal

print(f"Aplicando pesos: Texto ({PESO_TEXTO}), Imagen ({PESO_IMAGEN}), Vídeo ({PESO_VIDEO})")

df_ensemble['prob_final'] = (
    (df_ensemble['prob_texto'] * PESO_TEXTO) +
    (df_ensemble['prob_imagen'] * PESO_IMAGEN) +
    (df_ensemble['prob_video'] * PESO_VIDEO)
)

Aplicando pesos: Texto (0.7), Imagen (0.15), Vídeo (0.15)


In [17]:
# Predicción binaria final
df_ensemble['prediccion_ensemble'] = (df_ensemble['prob_final'] > 0.5).astype(int)

# 5. Evaluación del ensemble

In [18]:
y_true = df_ensemble['label']
y_pred = df_ensemble['prediccion_ensemble']

print("\n" + "="*50)
print("🏆 RESULTADOS DEL MODELO ENSEMBLE MULTIMODAL")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS DEL MODELO ENSEMBLE MULTIMODAL
F1-Score (Macro): 0.7098
Accuracy: 0.7131

Matriz de Confusión:
 [[206  55]
 [ 89 152]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.70      0.79      0.74       261
    Misógino       0.73      0.63      0.68       241

    accuracy                           0.71       502
   macro avg       0.72      0.71      0.71       502
weighted avg       0.72      0.71      0.71       502



In [19]:
# Guardar las predicciones completas del Ensemble Multimodal
ruta_ensemble_final = os.path.join(DIR_RESULTADOS, "predicciones_ensemble_final.csv")
df_ensemble.to_csv(ruta_ensemble_final, index=False)

print(f"✅ ¡Resultados completos del Ensemble guardados en: {ruta_ensemble_final}!")

✅ ¡Resultados completos del Ensemble guardados en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble/predicciones_ensemble_final.csv!


# 6. Análisis de errores

In [20]:
print("\n🔍 Generando archivo para el Análisis de Errores...")

# Filtrar donde el Ensemble se equivocó
df_errores = df_ensemble[df_ensemble['label'] != df_ensemble['prediccion_ensemble']].copy()

def clasificar_error(row):
    if row['label'] == 1 and row['prediccion_ensemble'] == 0:
        return "Falso Negativo (No lo detectó)"
    elif row['label'] == 0 and row['prediccion_ensemble'] == 1:
        return "Falso Positivo (Alarma falsa)"

df_errores['tipo_error'] = df_errores.apply(clasificar_error, axis=1)


🔍 Generando archivo para el Análisis de Errores...


In [21]:
# Ordenamos por la 'confianza' del modelo para ver los errores más graves primero
# Error grave = probabilidad muy lejana a 0.5 (ej. Falso Positivo con 0.99 de probabilidad)
df_errores['severidad_error'] = abs(df_errores['prob_final'] - 0.5)
df_errores = df_errores.sort_values(by=['tipo_error', 'severidad_error'], ascending=[True, False])

# Reordenar columnas para que sea fácil de leer en Excel
columnas_excel = [
    'id_EXIST', 'tipo_error', 'label', 'prediccion_ensemble', 'prob_final',
    'prob_texto', 'prob_imagen', 'prob_video', 'text'
]
df_errores = df_errores[columnas_excel]

In [22]:
ruta_errores = os.path.join(DIR_RESULTADOS, "casos_para_analisis_errores.xlsx")
df_errores.to_excel(ruta_errores, index=False)

print(f"✅ Se han encontrado {len(df_errores)} errores.")
print(f"📁 Archivo Excel de errores guardado en: {ruta_errores}")
print("¡Abre el Excel para analizar cualitativamente dónde falla tu arquitectura multimodal!")

✅ Se han encontrado 144 errores.
📁 Archivo Excel de errores guardado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble/casos_para_analisis_errores.xlsx
¡Abre el Excel para analizar cualitativamente dónde falla tu arquitectura multimodal!
